# Vinyl Label Reader + Discogs Search

Read a vinyl record label image using a vision LLM (via OpenRouter) to extract album name and artist, then search for all releases on **Discogs**.

**Setup:** Add your keys to `.env`:
```
DISCOGS_TOKEN=...
OPENROUTER_API_KEY=...
```

Change the vision model in `backend/config.py`.

In [7]:
import base64
import json
import os
import sys
from pathlib import Path

import pandas as pd
import requests
from dotenv import load_dotenv

load_dotenv(override=True)

# Import config from backend/
sys.path.insert(0, str(Path.cwd() / "backend"))
from config import DISCOGS_BASE_URL, DISCOGS_USER_AGENT, MAX_RANKING_RESULTS, OPENROUTER_URL, VISION_MODEL

DISCOGS_TOKEN = os.getenv("DISCOGS_TOKEN")
OPENROUTER_API_KEY = os.getenv("OPENROUTER_API_KEY")

if not DISCOGS_TOKEN:
    raise ValueError("DISCOGS_TOKEN not found in .env")
if not OPENROUTER_API_KEY:
    raise ValueError("OPENROUTER_API_KEY not found in .env")

DISCOGS_HEADERS = {"User-Agent": DISCOGS_USER_AGENT, "Authorization": f"Discogs token={DISCOGS_TOKEN}"}

LABEL_IMAGE_PATH = Path("img/labels/columbia-black-silvewer-early-1000px.jpg")

In [8]:
def llm_chat(messages: list[dict]) -> tuple[str, list[dict]]:
    """Send messages to the LLM and return (response_text, updated_messages)."""
    resp = requests.post(
        OPENROUTER_URL,
        headers={"Authorization": f"Bearer {OPENROUTER_API_KEY}", "Content-Type": "application/json"},
        json={"model": VISION_MODEL, "messages": messages},
    )
    resp.raise_for_status()
    content = resp.json()["choices"][0]["message"]["content"]
    messages.append({"role": "assistant", "content": content})
    return content, messages


def parse_json(raw: str) -> dict | list:
    cleaned = raw.strip().removeprefix("```json").removeprefix("```").removesuffix("```").strip()
    return json.loads(cleaned)


def read_label_image(image_path: Path) -> tuple[dict, list[dict]]:
    """Extract label metadata via vision. Returns (label_data, conversation_messages)."""
    image_bytes = image_path.read_bytes()
    b64_image = base64.b64encode(image_bytes).decode()
    mime_type = "image/jpeg" if image_path.suffix.lower() in (".jpg", ".jpeg") else "image/png"

    messages = [
        {
            "role": "user",
            "content": [
                {
                    "type": "image_url",
                    "image_url": {"url": f"data:{mime_type};base64,{b64_image}"},
                },
                {
                    "type": "text",
                    "text": (
                        "Look at this vinyl record label image. "
                        "Extract as much information as possible. "
                        "Return multiple possible variations ordered from most likely to least likely for albums and artists. "
                        "For albums: include the full title, shorter versions without subtitles, and any plausible variations. "
                        "For artists: include variations (e.g. with/without featured artists). "
                        "Also extract any optional metadata visible on the label. "
                        "Output ONLY a JSON object with these keys:\n"
                        '- "albums": array of strings (required)\n'
                        '- "artists": array of strings (required)\n'
                        '- "country": string or null (if visible, e.g. "Made in U.S.A.")\n'
                        '- "format": string or null (e.g. "LP", "45 RPM", "EP")\n'
                        '- "label": string or null (record label name, e.g. "Columbia", "RCA")\n'
                        '- "catno": string or null (catalog number)\n'
                        '- "year": string or null (release year if visible)\n'
                        "Nothing else."
                    ),
                },
            ],
        }
    ]

    raw, messages = llm_chat(messages)
    return parse_json(raw), messages


label_candidates, conversation = read_label_image(LABEL_IMAGE_PATH)
print(json.dumps(label_candidates, indent=2))

{
  "albums": [
    "THE BESSIE SMITH STORY - Volume I",
    "The Bessie Smith Story Vol. I",
    "The Bessie Smith Story Volume I",
    "The Bessie Smith Story"
  ],
  "artists": [
    "Bessie Smith with Louis Armstrong",
    "Bessie Smith"
  ],
  "country": "USA",
  "format": "LP",
  "label": "Columbia",
  "catno": "GL 503",
  "year": null
}


In [9]:
def discogs_search(max_pages: int = 10, **params) -> list[dict]:
    """Search Discogs with arbitrary params, paginating through all results."""
    params.setdefault("type", "release")
    params.setdefault("per_page", 50)
    all_results = []
    page = 1
    while page <= max_pages:
        params["page"] = page
        resp = requests.get(
            f"{DISCOGS_BASE_URL}/database/search",
            headers=DISCOGS_HEADERS,
            params=params,
        )
        resp.raise_for_status()
        data = resp.json()
        results = data.get("results", [])
        if not results:
            break
        all_results.extend(results)
        if page >= data.get("pagination", {}).get("pages", 1):
            break
        page += 1
    return all_results

In [10]:
candidate_albums = label_candidates["albums"]
candidate_artists = label_candidates["artists"]
label_meta = {
    k: label_candidates.get(k)
    for k in ("country", "format", "label", "catno", "year")
    if label_candidates.get(k)
}
print("Label metadata:", label_meta or "(none detected)")


def prefilter(releases: list[dict]) -> list[dict]:
    """Drop results where none of the candidate artist names appear in the title."""
    artists_lower = {a.lower() for a in candidate_artists}
    filtered = [
        r for r in releases
        if any(a in r.get("title", "").lower() for a in artists_lower)
    ]
    return filtered if filtered else releases


def search_with_strategy() -> tuple[list[dict], str]:
    """Try progressively looser search strategies. Returns (releases, strategy_description)."""

    # 1. Catalog number + label (most precise)
    if label_meta.get("catno"):
        params = {"catno": label_meta["catno"]}
        if label_meta.get("label"):
            params["label"] = label_meta["label"]
        results = discogs_search(**params)
        if results:
            return results, f"catno='{label_meta['catno']}'" + (
                f" + label='{label_meta['label']}'" if "label" in params else ""
            )

        if label_meta.get("label"):
            results = discogs_search(catno=label_meta["catno"])
            if results:
                return results, f"catno='{label_meta['catno']}'"

    # 2. Freeform query (q) with album + artist
    for album in candidate_albums:
        for artist in candidate_artists:
            results = discogs_search(q=f"{artist} {album}")
            if results:
                return results, f"q='{artist} {album}'"

    # 3. Strict release_title + artist
    for album in candidate_albums:
        for artist in candidate_artists:
            results = discogs_search(artist=artist, release_title=album)
            if results:
                return results, f"release_title='{album}' + artist='{artist}'"

    # 4. Artist-only search, fuzzy match titles
    for artist in candidate_artists:
        artist_results = discogs_search(artist=artist)
        if not artist_results:
            continue
        album_names_lower = {a.lower() for a in candidate_albums}
        matched = [
            r for r in artist_results
            if any(a in r.get("title", "").lower() for a in album_names_lower)
        ]
        if matched:
            return matched, f"artist='{artist}' + fuzzy title match"
        return artist_results, f"artist='{artist}' (all releases, no title match)"

    return [], "exhausted all strategies"


def rank_results(releases: list[dict], conv: list[dict]) -> tuple[list[int], list[int]]:
    """Ask the LLM to rank Discogs results using the same conversation context."""
    candidates = [
        {
            "index": i,
            "title": r.get("title"),
            "year": r.get("year"),
            "country": r.get("country"),
            "format": ", ".join(r.get("format", [])),
            "label": ", ".join(r.get("label", [])),
            "catno": r.get("catno"),
        }
        for i, r in enumerate(releases[:MAX_RANKING_RESULTS])
    ]

    conv.append({
        "role": "user",
        "content": (
            "I searched Discogs and found these candidate releases. "
            "Based on the vinyl label you just analyzed, rank them by how likely each one is the exact record shown in the image. "
            "Output ONLY a JSON object with two keys:\n"
            '- "likeliness": array of index values ordered from most likely to least likely\n'
            '- "discarded": array of index values for records that are FOR SURE not the correct record '
            "(e.g. completely wrong artist, wrong album, clearly a different release). "
            "Only discard records you are certain about.\n\n"
            f"{json.dumps(candidates, indent=2)}"
        ),
    })

    raw, _ = llm_chat(conv)
    result = parse_json(raw)
    return result.get("likeliness", []), result.get("discarded", [])


# Search
releases, strategy = search_with_strategy()
print(f"Strategy: {strategy}")
print(f"Results: {len(releases)}")

if releases:
    # Pre-filter obvious mismatches
    releases = prefilter(releases)
    print(f"After pre-filter: {len(releases)}")

    # LLM ranking (same conversation)
    likeliness, discarded = rank_results(releases, conversation)
    print(f"LLM likeliness order: {likeliness}")
    print(f"LLM discarded: {discarded}")

    # Reorder: likeliness first, then remaining, excluding discarded
    discarded_set = set(discarded)
    seen = set()
    ordered = []
    for idx in likeliness:
        if idx not in discarded_set and 0 <= idx < len(releases) and idx not in seen:
            ordered.append(releases[idx])
            seen.add(idx)
    for idx, r in enumerate(releases):
        if idx not in seen and idx not in discarded_set:
            ordered.append(r)
            seen.add(idx)
    releases = ordered

    top = releases[0]
    print(f"\nBest match: {top.get('title')}")
    print(f"  catno={top.get('catno')}, country={top.get('country')}, year={top.get('year')}")

Label metadata: {'country': 'USA', 'format': 'LP', 'label': 'Columbia', 'catno': 'GL 503'}
Strategy: catno='GL 503' + label='Columbia'
Results: 2
After pre-filter: 2
LLM likeliness order: [1, 0]
LLM discarded: []

Best match: Bessie Smith With Louis Armstrong - The Bessie Smith Story - Volume 1
  catno=GL 503, country=US, year=1951


In [11]:
from IPython.display import display

if not releases:
    print("No releases found — nothing to display.")
    df = pd.DataFrame()
else:
    df = pd.DataFrame([
        {
            "title": r.get("title"),
            "year": r.get("year"),
            "country": r.get("country"),
            "format": ", ".join(r.get("format", [])),
            "label": ", ".join(r.get("label", [])),
            "catno": r.get("catno"),
            "discogs_url": f"https://www.discogs.com{r.get('uri', '')}",
            "cover_image": r.get("cover_image"),
        }
        for r in releases
    ])

    df["year"] = pd.to_numeric(df["year"], errors="coerce").astype("Int64")
    df.sort_values("year", inplace=True, na_position="last")
    df.reset_index(drop=True, inplace=True)
    display(df)

,title,year,country,format,label,catno,discogs_url,cover_image
0,Bessie Smith With Louis Armstrong - The Bessie...,1951,US,"Vinyl, LP, Compilation, Mono","Columbia, Columbia Records, Inc., The Golden E...",GL 503,https://www.discogs.com/release/3458098-Bessie...,https://i.discogs.com/KelpmiqL3y6XUjIgcNxshybl...
1,Bessie Smith With Louis Armstrong - The Bessie...,<NA>,Canada,"Vinyl, LP, Compilation, Mono","Columbia, Columbia Masterworks",GL 503,https://www.discogs.com/release/11674868-Bessi...,https://i.discogs.com/uIJCjKWKkL1uFCvqVvxm1z52...


In [12]:
# Quick stats
if not df.empty:
    print(f"Total releases: {len(df)}")
    print(f"Countries: {df['country'].nunique()}")
    print(f"Year range: {df['year'].min()} - {df['year'].max()}")
    print(f"\nFormats:\n{df['format'].value_counts().to_string()}")

Total releases: 2
Countries: 2
Year range: 1951 - 1951

Formats:
format
Vinyl, LP, Compilation, Mono    2


---

# Batch Run Analysis

Query MongoDB for batch processing results and analyze success/failure rates.

In [ ]:
from pymongo import MongoClient

mongo_uri = os.getenv("MONGO_URI", "mongodb://localhost:27017")
mongo_db = os.getenv("MONGO_DB", "vinyl_recko")
client = MongoClient(mongo_uri)
db = client[mongo_db]

# Overview
batches = list(db.batches.find({}, {"_id": 0}))
items = list(db.batch_items.find({}, {"_id": 0}))

print(f"Batches: {len(batches)}")
for b in batches:
    print(f"  {b['batch_id'][:8]}... — {b['status']}, {b['processed']}/{b['total_images']} processed, {b['failed']} failed")

print(f"\nTotal items: {len(items)}")
status_counts = pd.Series([i["status"] for i in items]).value_counts()
review_counts = pd.Series([i["review_status"] for i in items]).value_counts()
print(f"\nBy processing status:\n{status_counts.to_string()}")
print(f"\nBy review status:\n{review_counts.to_string()}")

In [ ]:
# Strategy distribution
strategies = [i.get("strategy", "N/A") for i in items if i["status"] == "completed"]
strategy_series = pd.Series(strategies)

# Simplify strategy names for grouping
def simplify_strategy(s):
    if not s:
        return "none"
    if s.startswith("catno="):
        return "catno"
    if s.startswith("q="):
        return "freeform (q)"
    if s.startswith("release_title="):
        return "release_title + artist"
    if "fuzzy" in s:
        return "artist + fuzzy title"
    if "all releases" in s:
        return "artist only (no title match)"
    return s

strategy_groups = strategy_series.map(simplify_strategy).value_counts()
print("Strategy distribution (completed items):")
print(strategy_groups.to_string())
print(f"\nTotal completed: {len(strategies)}")

In [ ]:
# Problematic items: dismissed, no results, errored
problems = []
for i in items:
    result_count = len(i.get("results") or [])
    label = i.get("label_data") or {}
    artists = ", ".join(label.get("artists", []))
    albums = ", ".join(label.get("albums", []))
    row = {
        "filename": i["image_filename"],
        "status": i["status"],
        "review": i["review_status"],
        "strategy": simplify_strategy(i.get("strategy")),
        "results": result_count,
        "artists": artists,
        "albums": albums,
        "error": i.get("error"),
    }
    if i["review_status"] == "skipped" or result_count == 0 or i["status"] == "error":
        problems.append(row)

problems_df = pd.DataFrame(problems)
print(f"Problematic items: {len(problems_df)} / {len(items)}")
display(problems_df[["filename", "status", "review", "strategy", "results", "artists", "albums", "error"]])

In [ ]:
# Telemetry: search_records timing
records = list(db.search_records.find({}, {"_id": 0}))
if records:
    records_df = pd.DataFrame(records)
    batch_records = records_df[records_df["batch_id"].notna()]
    single_records = records_df[records_df["batch_id"].isna()]

    print(f"Search records: {len(records_df)} total ({len(batch_records)} batch, {len(single_records)} single)")
    print(f"\nBatch timing (ms):")
    if not batch_records.empty:
        print(f"  mean: {batch_records['total_duration_ms'].mean():.0f}")
        print(f"  median: {batch_records['total_duration_ms'].median():.0f}")
        print(f"  min: {batch_records['total_duration_ms'].min():.0f}")
        print(f"  max: {batch_records['total_duration_ms'].max():.0f}")
    print(f"\nBatch status distribution:")
    print(batch_records["status"].value_counts().to_string())
else:
    print("No search records found.")